In [1]:
%matplotlib widget

from blackjack_py import ProbabilisticRankShoe


from blackjack.blackjack_round import BJRound, BJStage, BJRules
from blackjack.actions import PlayerAction, DealerAction
from blackjack.cards import Card, Rank
import numpy as np
import time
from datetime import datetime
import os
import tqdm
from collections import deque
# from blackjack.shoe import ProbabilisticRankShoe
import time
from blackjack.floor_ceil_node import (
    FloorCeilNode, SplitNode, DecisionNode, DealerCheckBJNode
)
from blackjack.dealer_sim import run_dealer_cards_simulation
import datetime
import matplotlib.pyplot as plt
from blackjack.tree_utils import iterate_nodes_by_levels
import os
from blackjack import dealer_sim

In [2]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing
from dataclasses import asdict

    
# Import inside worker to ensure proper initialization
from blackjack.abstract_node import ValueNode
from blackjack_py import ProbabilisticRankShoe
from blackjack.blackjack_round import BJRound, BJRules
from blackjack.floor_ceil_node import DecisionNode


def build_root_node(bj_round, shoe, n_depth):
    stage = bj_round.get_stage()

    if bj_round.get_stage() == BJStage.DEALER_CHECK_BJ:
        # dealer checks blackjack with ten
        # insurance not offered
        root_node = DealerCheckBJNode(
            bj_round,
            shoe,
            max_hand_size_full_enum=1,
            took_insurance=False,
            insurance_offered=False,
            dealer_sim_depth=n_depth,
            sim_algo="combo"
        )
    elif bj_round.get_stage() == BJStage.DEALER_CARD \
        and len(bj_round.player_hands) == 1 \
        and bj_round.player_hands[0].is_natural_blackjack():
        # player has blackjack, insurance not offered - go to dealer card immediately
        root_node = ValueNode(
            bj_round.bet_unit * bj_round.rules.natural_blackjack_payout,
            bj_round=bj_round,
            shoe=shoe
        )
        root_node.bj_round = bj_round
    else:
        # insurance or a normal game node
        root_node = DecisionNode(
            bj_round,
            shoe,
            max_hand_size_full_enum=1,
            dealer_sim_depth=n_depth,
            sim_algo="combo"
        )
            
    return root_node


def build_tree(root_node, gap_target):
    if isinstance(root_node, ValueNode):
        return
    
    gap_target_unit = gap_target * root_node.bj_round.bet_unit
    root_node.build_tree()
    for i in range(100):
        root_node.convert_to_full_up_to_depth(depth=i)
        if root_node.get_ceil_value() - root_node.get_floor_value() < gap_target_unit:
            return
    raise RuntimeError("Failed to converge")





In [3]:
rules = BJRules(
    dealer_checks_blackjack=True,
    dealer_hits_soft_17=False,
    allow_late_surrender=False,
    allow_early_surrender_on_ten=False,
    allow_early_surrender_on_ace=False,
    allow_early_surrender_on_all=False,
    dealer_shows_card_on_surrender=False,
    allow_insurance_vs_ace=True,
    natural_blackjack_payout=3/2,
    surrender_payout=1/2,
    insurance_payout=2/1,
    max_splits_allowed=1,
    allow_action_on_split_aces=True,
    allow_double_after_split=True,
    allow_double_on_soft=True,
    allow_split_different_tens=True
)

In [4]:
def generate_initial_rounds():
    for player_card_0 in range(2, 12):
        for player_card_1 in range(player_card_0, 12):
            for dealer_upcard in range(2, 12):
                n_count = 1 if player_card_0 == player_card_1 else 2
                yield player_card_0, player_card_1, dealer_upcard, n_count

In [6]:
nodes = []
probs = []
# for p0 in range(2, 12):
#     for p1 in range(p0, 12):
#         for d in range(2, 12):


for p0, p1, d, n in generate_initial_rounds():
    bj_round = BJRound(rules)
    bj_round.start_round(100)
    shoe = ProbabilisticRankShoe.seeded(6, 42)
    prob = 1
    for c in [p0, p1, d]:
        prob_c_dict = shoe.get_rank_value_probabilities()
        prob *= prob_c_dict[c]
        bj_round.take_card(c)
        shoe.burn_rank_value(c)
    root_node = build_root_node(
        bj_round,
        shoe,
        5
    )
    nodes.append(root_node)
    probs.append(prob * n)

In [7]:
np.sum(probs)

np.float64(1.0)

In [8]:
# node_values = []
# node_ceil_values = []
# node_floor_values = []

# for root_node in tqdm.tqdm(nodes):
#     build_tree(root_node, gap_target=0.1)
#     node_values.append(root_node.get_value())
#     node_ceil_values.append(root_node.get_ceil_value())
#     node_floor_values.append(root_node.get_floor_value())

In [ ]:
def get_start_shoe():
    shoe = ProbabilisticRankShoe.seeded(6, 42)
    return shoe


def process_single_round(args):
    """Worker function that reconstructs node and computes value."""
    p0, p1, d, gap_target, rules_dict = args
    
    # Reconstruct rules and objects
    rules = BJRules(**rules_dict)
    shoe = get_start_shoe()
    bj_round = BJRound(rules)
    bj_round.start_round(100)
    
    for c in [p0, p1, d]:
        bj_round.take_card(c)
        shoe.burn_rank_value(d)
    
    root_node = build_root_node(bj_round, shoe, n_depth=6)
    build_tree(root_node, gap_target)

    return root_node.get_value(), root_node.get_floor_value(), root_node.get_ceil_value()


max_workers = multiprocessing.cpu_count()
gap_target = 0.001

rules_dict = asdict(rules)

# Build task list with serializable args
tasks = []
probs = []
for p0, p1, d, n in generate_initial_rounds():
    shoe = get_start_shoe()
    prob = 1
    for c in [p0, p1, d]:
        prob_c_dict = shoe.get_rank_value_probabilities()
        prob *= prob_c_dict[c]
        shoe.burn_rank_value(c)
    tasks.append((p0, p1, d, gap_target, rules_dict))
    probs.append(n * prob)

ev_by_nodes = [None] * len(tasks)

with ProcessPoolExecutor(max_workers=max_workers) as executor:
    future_to_idx = {
        executor.submit(process_single_round, task): idx 
        for idx, task in enumerate(tasks)
    }
    
    for future in tqdm.tqdm(as_completed(future_to_idx), total=len(tasks)):
        idx = future_to_idx[future]
        ev_by_nodes[idx] = future.result()

100%|██████████| 550/550 [02:29<00:00,  3.67it/s]


In [10]:
ev_mean = np.sum(np.array(ev_by_nodes) * np.array(probs)[:, np.newaxis], axis=0)

In [11]:
ev_mean

array([-2.89633446, -2.89638829, -2.8874303 ])

In [12]:
for node, (ev, ev_min, ev_max) in zip(nodes, ev_by_nodes):
    print("=" * 20)
    print(str(node.bj_round))
    print(f"value, value_min, value_max = {ev:.2f}, {ev_min:.2f}, {ev_max:.2f}")
    print("=" * 20)

Last card: 2
Dealer 2(2)
Player 2,2(4)[$100]
value, value_min, value_max = -14.45, -14.45, -14.40
Last card: 3
Dealer 3(3)
Player 2,2(4)[$100]
value, value_min, value_max = -10.55, -10.56, -10.50
Last card: 4
Dealer 4(4)
Player 2,2(4)[$100]
value, value_min, value_max = -5.90, -5.90, -5.86
Last card: 5
Dealer 5(5)
Player 2,2(4)[$100]
value, value_min, value_max = 2.60, 2.60, 2.61
Last card: 6
Dealer 6(6)
Player 2,2(4)[$100]
value, value_min, value_max = 5.90, 5.90, 5.90
Last card: 7
Dealer 7(7)
Player 2,2(4)[$100]
value, value_min, value_max = -7.77, -7.77, -7.77
Last card: 8
Dealer 8(8)
Player 2,2(4)[$100]
value, value_min, value_max = -16.14, -16.14, -16.13
Last card: 9
Dealer 9(9)
Player 2,2(4)[$100]
value, value_min, value_max = -24.01, -24.01, -23.99
Last card: 10
Dealer 10(10)
Player 2,2(4)[$100]
value, value_min, value_max = -33.91, -33.91, -33.89
Last card: 11
Dealer A(1/11)
Player 2,2(4)[$100]
value, value_min, value_max = -50.84, -50.84, -50.83
Last card: 2
Dealer 2(2)
Player

In [13]:
ev

2.8810556606684905